In [1]:
# Instala o LTNtorch apenas se ainda não estiver instalado (útil no Google Colab)

import importlib.util, subprocess, sys
if importlib.util.find_spec("ltn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/tommasocarraro/LTNtorch.git"])
print("LTNtorch pronto.")


LTNtorch pronto.


In [2]:
#!/usr/bin/env python
# coding: utf-8

# In[32]:
import torch
import torch.nn as nn
import ltn
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# In[33]:
# Configure o LTNtorch para usar o dispositivo apropriado.
ltn.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {ltn.device}")

Using device: cpu


In [4]:
# In[34]:
# 1. DEFINIR CONECTIVOS LÓGICOS E QUANTIFICADORES (REQUERIDOS NO LTNTORCH)
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
sat_agg = ltn.fuzzy_ops.SatAgg()

In [5]:
# In[35]:
# 2. CRIAR OBJETOS LTN (Constantes e Variáveis)
red_ball = ltn.Constant(torch.tensor([1.0, 0.0, 0.0]), trainable=False)
blue_cube = ltn.Constant(torch.tensor([0.0, 0.0, 1.0]), trainable=False)

balls = ltn.Variable("balls", torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.5, 0.0, 0.5],
]))
cubes = ltn.Variable("cubes", torch.tensor([
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 0.0],
    [0.5, 0.5, 0.5],
]))

In [6]:
# In[36]:
# 3. DEFINIR PREDICADOS (Propriedades/relações lógicas)
class ColorPredicate(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(3, 10),
            nn.ReLU(),
            nn.Linear(10, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        if hasattr(x, 'value'):
            x = x.value
        return self.network(x).squeeze()

In [7]:
is_red = ltn.Predicate(ColorPredicate())
is_blue = ltn.Predicate(ColorPredicate())
is_round = ltn.Predicate(ColorPredicate())

In [8]:
# 4. DEFINIR AXIOMAS LÓGICOS (Base de conhecimento)
def axioms():
    facts = [
        is_red(red_ball),
        is_blue(blue_cube),
        is_round(red_ball),
        Not(is_round(blue_cube))
    ]
    rules = [
        Forall(balls, is_round(balls)),
        Forall(ltn.Variable("x", torch.randn(10, 3, device=ltn.device)),
            Implies(is_red(ltn.Variable("x", torch.randn(10, 3, device=ltn.device))),
            is_round(ltn.Variable("x", torch.randn(10, 3, device=ltn.device)))))
    ]
    all_axioms = facts + rules
    return sat_agg(*all_axioms)

In [9]:
# 5. VERIFICAÇÃO DE SATISFATÓRIA (Raciocínio)
def check_satisfiability():
    sat_level = axioms()
    print(f"Knowledge base satisfaction level: {sat_level.item():.4f}")
    return sat_level

In [10]:
# 6. TREINAMENTO DOS PREDICADOS (Aprendendo com os dados)
def train_predicates(epochs=100):
    parameters = list(is_red.model.parameters()) + \
                 list(is_blue.model.parameters()) + \
                 list(is_round.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)
    print("\n=== TRAINING PREDICATES ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat_loss = 1 - axioms()
        sat_loss.backward()
        optimizer.step()
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Satisfaction = {1-sat_loss.item():.4f}")

In [11]:
# In[43]:
# 7. QUERYING THE KNOWLEDGE BASE (Inference)
def query_knowledge_base():
    print("\n=== QUERYING KNOWLEDGE BASE ===")
    # Query 1: Is the red ball red?
    query1 = is_red(red_ball)
    print(f"Q1: Is the red ball red? A: {query1.value.item():.4f}")

    # Query 2: Is the blue cube round?
    query2 = is_round(blue_cube)
    print(f"Q2: Is the blue cube round? A: {query2.value.item():.4f}")

    # Query 3: Are all balls round? (Universal quantification)
    all_balls_round = Forall(balls, is_round(balls))
    print(f"Q3: Are all balls round? A: {all_balls_round.value.item():.4f}")

In [12]:
# 8. EXECUTE O EXEMPLO COMPLETO
print("=== EXEMPLO DE RACIOCÍNIO LÓGICO LTN ===")
initial_sat = check_satisfiability()
train_predicates(epochs=100)
final_sat = check_satisfiability()
query_knowledge_base()
print("\n=== PRINCIPAIS INFORMAÇÕES SOBRE O LTNObjects ===")
print("1. LTNObjects encapsulam tensores com significado lógico.")
print("2. Constantes representam entidades específicas")
print("3. As variáveis representam conjuntos quantificáveis")
print("4. Os predicados mapeiam LTNObjects para valores de verdade [0,1]")
print("5. Os conectivos lógicos (E, Ou, Não, Implica) operam sobre valores de verdade.")
print("6. Os quantificadores (Forall, Exists) agregam sobre a variávels")
print("7. No LTNTorch, você deve definir explicitamente todos os conectivos lógicos.")
print("8. A função SatAgg combina múltiplos axiomas em uma única pontuação de satisfação.")


=== EXEMPLO DE RACIOCÍNIO LÓGICO LTN ===
Knowledge base satisfaction level: 0.5544

=== TRAINING PREDICATES ===
Epoch 0: Satisfaction = 0.5537
Epoch 20: Satisfaction = 0.6988
Epoch 40: Satisfaction = 0.8375
Epoch 60: Satisfaction = 0.8250
Epoch 80: Satisfaction = 0.8284
Knowledge base satisfaction level: 0.9394

=== QUERYING KNOWLEDGE BASE ===
Q1: Is the red ball red? A: 0.9401
Q2: Is the blue cube round? A: 0.0759
Q3: Are all balls round? A: 0.9346

=== PRINCIPAIS INFORMAÇÕES SOBRE O LTNObjects ===
1. LTNObjects encapsulam tensores com significado lógico.
2. Constantes representam entidades específicas
3. As variáveis representam conjuntos quantificáveis
4. Os predicados mapeiam LTNObjects para valores de verdade [0,1]
5. Os conectivos lógicos (E, Ou, Não, Implica) operam sobre valores de verdade.
6. Os quantificadores (Forall, Exists) agregam sobre a variávels
7. No LTNTorch, você deve definir explicitamente todos os conectivos lógicos.
8. A função SatAgg combina múltiplos axiomas em

In [13]:
# ==============================================================================
# EXTENSÃO: DIFERENTES FORMAS DE IGUALDADE EM LTN (Extensão Original)
# ==============================================================================
# In[45]:
print("\n" + "="*50)
print("EXTENSION: DIFFERENT FORMS OF EQUALITY IN LTN")
print("="*50)


EXTENSION: DIFFERENT FORMS OF EQUALITY IN LTN


In [14]:
Not_eq = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And_eq = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or_eq = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies_eq = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
Forall_eq = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists_eq = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")
sat_agg_eq = ltn.fuzzy_ops.SatAgg()

In [15]:
red_object1 = ltn.Constant(torch.tensor([1.0, 0.0, 0.0], device=ltn.device), trainable=True)
red_object2 = ltn.Constant(torch.tensor([0.95, 0.05, 0.0], device=ltn.device), trainable=True)
blue_object = ltn.Constant(torch.tensor([0.0, 0.0, 1.0], device=ltn.device), trainable=True)

objects = ltn.Variable("objetos", torch.tensor([
    [1.0, 0.0, 0.0],
    [0.95, 0.05, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, 1.0, 0.0],
], device=ltn.device))
print("Experimento sobre igualdade: configuração concluída!")

# In[46]: 1. Igualdade diagonal LTN
print("\n" + "-"*30)
print("1. IGUALDADE DIAGONAL LTN")
print("-"*30)

Experimento sobre igualdade: configuração concluída!

------------------------------
1. IGUALDADE DIAGONAL LTN
------------------------------


In [16]:
class DiagonalEquality(nn.Module):
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        cos_sim = torch.nn.functional.cosine_similarity(x_val, y_val, dim=-1)
        return 0.5 * (cos_sim + 1.0)

In [17]:
# Teste da Igualdade diagonal euclidiana
Equal_Diag = ltn.Predicate(DiagonalEquality().to(ltn.device))

In [18]:
def axioms_withEquality_Diag():
    axioms = [
        Forall_eq(objects, Equal_Diag(objects, objects)),
        Equal_Diag(red_object1, red_object2),
        Equal_Diag(red_object2, red_object1),
        Implies_eq(And_eq(Equal_Diag(red_object1, red_object2), Equal_Diag(red_object2, red_object1)), Equal_Diag(red_object1, red_object1)),
        Not_eq(Equal_Diag(red_object1, blue_object))
    ]
    return sat_agg_eq(*axioms)

In [19]:
print("Testando a igualdade diagonal...")
sat_diag = axioms_withEquality_Diag()
print(f"Satisfação KB de igualdade diagonal: {sat_diag.item():.4f}")
print(f"P: Os dois objetos vermelhos são iguais? R: {Equal_Diag(red_object1, red_object2).value.item():.4f}")
print(f"P: Um objeto vermelho é igual a um objeto azul? R: {Equal_Diag(red_object1, blue_object).value.item():.4f}")

# In[50]: 2. Igualdade de cosseno
print("\n" + "-"*30)
print("2. IGUALDADE DE COSSENO.")
print("-"*30)

Testando a igualdade diagonal...
Satisfação KB de igualdade diagonal: 0.7764
P: Os dois objetos vermelhos são iguais? R: 0.9993
P: Um objeto vermelho é igual a um objeto azul? R: 0.5000

------------------------------
2. IGUALDADE DE COSSENO.
------------------------------


In [21]:
class CosineEquality(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.tensor(1.0, device=ltn.device))
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        cos_sim = torch.nn.functional.cosine_similarity(x_val, y_val, dim=-1)
        return torch.sigmoid(self.temperature * (cos_sim + 1.0) / 2.0)

Equal_Cos = ltn.Predicate(CosineEquality().to(ltn.device))

In [22]:
def axioms_withEquality_Cos():
    parameters = list(Equal_Cos.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.1)
    print("Treinamento da Igualdade do Cosseno...")
    for epoch in range(200):
        optimizer.zero_grad()
        axioms = [Forall_eq(objects, Equal_Cos(objects, objects)), Equal_Cos(red_object1, red_object2), Equal_Cos(red_object2, red_object1), Not_eq(Equal_Cos(red_object1, blue_object))]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Época {epoch}: Satisfatibilidade da igualdade do cosseno = {sat.item():.4f}, Temp = {Equal_Cos.model.temperature.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Cos(objects, objects)), Equal_Cos(red_object1, red_object2), Not_eq(Equal_Cos(red_object1, blue_object)))

sat_cos = axioms_withEquality_Cos()
print(f"Satisfatibilidade KB da Igualdade do Cosseno Final: {sat_cos.item():.4f}")
print(f"Q: Os dois objetos vermelhos são iguais (cosseno)? A: {Equal_Cos(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (cosseno)? A: {Equal_Cos(red_object1, blue_object).value.item():.4f}")


Treinamento da Igualdade do Cosseno...
Época 0: Satisfatibilidade da igualdade do cosseno = 0.6112, Temp = 1.1000
Época 50: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5738
Época 100: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5680
Época 150: Satisfatibilidade da igualdade do cosseno = 0.6255, Temp = 1.5687
Satisfatibilidade KB da Igualdade do Cosseno Final: 0.5793
Q: Os dois objetos vermelhos são iguais (cosseno)? A: 0.8275
Q: O objeto vermelho é igual ao objeto azul (cosseno)? A: 0.6866


In [23]:
# In[51]: 3. Igualdade Euclidiana
print("\n" + "-"*30)
print("3. IGUALDADE EUCLIDIANA")
print("-"*30)


------------------------------
3. IGUALDADE EUCLIDIANA
------------------------------


In [24]:
class EuclideanEquality(nn.Module):
    def __init__(self):
        super().__init__()
        self.gamma = nn.Parameter(torch.tensor(0.5, device=ltn.device))
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        distance = torch.sum(torch.square(x_val - y_val), dim=-1)
        return torch.exp(-self.gamma * distance)

Equal_Eucl = ltn.Predicate(EuclideanEquality().to(ltn.device))


In [25]:
def axioms_withEquality_Eucl():
    parameters = list(Equal_Eucl.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.1)
    print("Treinamento em Igualdade Euclidiana...")
    for epoch in range(150):
        optimizer.zero_grad()
        axioms = [Forall_eq(objects, Equal_Eucl(objects, objects)), Equal_Eucl(red_object1, red_object2), Equal_Eucl(red_object2, red_object1), Not_eq(Equal_Eucl(red_object1, blue_object))]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 30 == 0:
            print(f"Época {epoch}: Satisfatibilidade da Igualdade Euclidiana = {sat.item():.4f}, Gamma = {Equal_Eucl.model.gamma.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Eucl(objects, objects)), Equal_Eucl(red_object1, red_object2), Not_eq(Equal_Eucl(red_object1, blue_object)))

sat_eucl = axioms_withEquality_Eucl()
print(f"Satisfatibilidade final da igualdade euclidiana KB: {sat_eucl.item():.4f}")
print(f"Q: Os dois objetos vermelhos são iguais (euclidianos)? A: {Equal_Eucl(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (euclidiano)? A: {Equal_Eucl(red_object1, blue_object).value.item():.4f}")


Treinamento em Igualdade Euclidiana...
Época 0: Satisfatibilidade da Igualdade Euclidiana = 0.8160, Gamma = 0.6000
Época 30: Satisfatibilidade da Igualdade Euclidiana = 0.9903, Gamma = 2.2782
Época 60: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4738
Época 90: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4658
Época 120: Satisfatibilidade da Igualdade Euclidiana = 0.9905, Gamma = 2.4479
Satisfatibilidade final da igualdade euclidiana KB: 0.9917
Q: Os dois objetos vermelhos são iguais (euclidianos)? A: 0.9879
Q: O objeto vermelho é igual ao objeto azul (euclidiano)? A: 0.0076


In [26]:
# In[52]: 4. Igualdade de Variedades Aprendíveis
print("\n" + "-"*30)
print("4. IGUALDADE DE VARIEDADES APRENDÍVEIS")
print("-"*30)

class LearnableEquality(nn.Module):
    def __init__(self, input_dim=3):
        super(LearnableEquality, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim * 2, 64), nn.ELU(),
            nn.Linear(64, 32), nn.ELU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'valor') else x
        y_val = y.value if hasattr(y, 'valor') else y
        cat_inputs = torch.cat([x_val, y_val], dim=-1)
        return self.net(cat_inputs).squeeze()

Equal_Learned = ltn.Predicate(LearnableEquality(input_dim=3).to(ltn.device))



------------------------------
4. IGUALDADE DE VARIEDADES APRENDÍVEIS
------------------------------


In [27]:
def axioms_withEquality_Learned():
    parameters = list(Equal_Learned.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)
    print("Treinaento da Rede de Igualdade aprodível...")
    for epoch in range(300):
        optimizer.zero_grad()
        axioms = [
            Forall_eq(objects, Equal_Learned(objects, objects)),
            Equal_Learned(red_object1, red_object2), Equal_Learned(red_object2, red_object1),
            Not_eq(Equal_Learned(red_object1, blue_object)), Not_eq(Equal_Learned(blue_object, red_object1)),
            Implies_eq(And_eq(Equal_Learned(red_object1, red_object2), Equal_Learned(red_object2, red_object1)), Equal_Learned(red_object1, red_object1))
        ]
        sat = sat_agg_eq(*axioms)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Satisfatibilidade com a igualdade aprendível= {sat.item():.4f}")
    return sat_agg_eq(Forall_eq(objects, Equal_Learned(objects, objects)), Equal_Learned(red_object1, red_object2), Not_eq(Equal_Learned(red_object1, blue_object)))

sat_learned = axioms_withEquality_Learned()
print(f"Satisfação final com a base de conhecimento sobre igualdade aprendível: {sat_learned.item():.4f}")
print(f"Q:Os dois objetos vermelhos são iguais (Aprendido)? A: {Equal_Learned(red_object1, red_object2).value.item():.4f}")
print(f"Q: O objeto vermelho é igual ao objeto azul (aprendido)? A: {Equal_Learned(red_object1, blue_object).value.item():.4f}")
print(f"Q: O objeto vermelho é igual a si mesmo (Aprendido)? A: {Equal_Learned(red_object1, red_object1).value.item():.4f}")


Treinaento da Rede de Igualdade aprodível...
Epoch 0: Satisfatibilidade com a igualdade aprendível= 0.5533
Epoch 50: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 100: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 150: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 200: Satisfatibilidade com a igualdade aprendível= 0.7959
Epoch 250: Satisfatibilidade com a igualdade aprendível= 0.7959
Satisfação final com a base de conhecimento sobre igualdade aprendível: 0.7113
Q:Os dois objetos vermelhos são iguais (Aprendido)? A: 1.0000
Q: O objeto vermelho é igual ao objeto azul (aprendido)? A: 0.0000
Q: O objeto vermelho é igual a si mesmo (Aprendido)? A: 1.0000


In [28]:
torch.manual_seed(42)  # reprodutibilidade: mesmos resultados a cada execução
# ==============================================================================
# NOVA EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D (Para os Alunos)
# ==============================================================================
# In[53]:
print("\n" + "="*60)
print("EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D")
print("="*60)

# ==============================================================================
# DESAFIO PARA OS ALUNOS:
# Nesta extensão, você é convidado a formalizar relações geométricas
# usando Lógica de Primeira Ordem (FOL) e implementá-las no LTN.
#
# Definimos três classes de figuras geométricas 2D baseadas no centro (x,y):
# - Circle(x): A figura x é um círculo
# - Square(x): A figura x é um quadrado
# - Triangle(x): A figura x é um triângulo
#
# E relações espaciais entre pares de figuras:
# - LeftOf(x,y): A figura x está à esquerda da figura y
# - Above(x,y): A figura x está acima da figura y
# - Overlaps(x,y): A figura x se sobrepõe à figura y
#
# AXIOMAS EM LÓGICA DE PRIMEIRA ORDEM (Base de Conhecimento):
# 1. Todos os círculos estão à esquerda de todos os quadrados:
#    ∀x ∀y (Circle(x) ∧ Square(y) → LeftOf(x,y))
#
# 2. Todos os triângulos estão acima de todos os círculos:
#    ∀x ∀y (Triangle(x) ∧ Circle(y) → Above(x,y))
#
# 3. A sobreposição é simétrica:
#    ∀x ∀y (Overlaps(x,y) → Overlaps(y,x))
#
# 4. Nenhum objeto se sobrepõe a si mesmo:
#    ∀x ¬Overlaps(x,x)
#
# 5. (ANCORAGEM) Axiomas de rotulagem: cada figura de treino é de uma classe
#    ∀c Circle(c), ∀c ¬Square(c), ∀s Square(s), ... e ∀s∀c ¬LeftOf(s,c)
#    (sem isso os classificadores colapsam para 0 e as implicações ficam
#     verdadeiras de forma vazia)
# 6. (EXTRA) Quadrados e triângulos não se sobrepõem:
#    ∀x ∀y (Square(x) ∧ Triangle(y) → ¬Overlaps(x,y))
#
# TAREFA DO ALUNO: Estenda esta base de conhecimento com novos axiomas! Ex:
# - ∀x ∀y (Square(x) ∧ Triangle(y) → ¬Overlaps(x,y))
# - ∀x (Circle(x) → ∃y (Square(y) ∧ LeftOf(x,y)))
# ==============================================================================

# 1. GERAÇÃO DE DADOS (Plano 2D Aleatório)
# Regiões: círculos x[0,4] y[0,5]; quadrados x[6,10] y[0,5]; triângulos x[0,10] y[6,10].
# Os quadrados ficam com y em [0,5] para NÃO invadirem a região dos triângulos
# (antes havia sobreposição em x[6,10], y[6,10] e a rede não conseguia separar as classes).
# Círculos agora com y em [0,5], para ficarem abaixo dos triângulos (y em [6,10])
def generate_random_shapes(num_per_shape=10):
    circles = torch.rand(num_per_shape, 2, device=ltn.device) * torch.tensor([4.0, 5.0], device=ltn.device)
    squares = torch.rand(num_per_shape, 2, device=ltn.device) * torch.tensor([4.0, 5.0], device=ltn.device) + torch.tensor([6.0, 0.0], device=ltn.device)
    triangles = torch.rand(num_per_shape, 2, device=ltn.device) * torch.tensor([10.0, 4.0], device=ltn.device) + torch.tensor([0.0, 6.0], device=ltn.device)
    return circles, squares, triangles

# 2. PREDICADOS NEURAIS
class ShapeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x):
        x_val = x.value if hasattr(x, 'value') else x
        return self.net(x_val).squeeze()

class LearnableSpatialRelation(nn.Module):
    def __init__(self, feature_extractor):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )

    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y

        features = self.feature_extractor(x_val, y_val)
        return self.net(features).squeeze(-1)


def left_of_features(x, y):
    return (y[..., 0] - x[..., 0]).unsqueeze(-1)

def above_features(x, y):
    return (x[..., 1] - y[..., 1]).unsqueeze(-1)

def distance_features(x, y):
    return torch.norm(x - y, dim=-1, keepdim=True)

is_circle = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_square = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_triangle = ltn.Predicate(ShapeClassifier().to(ltn.device))
left_of = ltn.Predicate(LearnableSpatialRelation(left_of_features).to(ltn.device))
above = ltn.Predicate(LearnableSpatialRelation(above_features).to(ltn.device))
overlaps = ltn.Predicate(LearnableSpatialRelation(distance_features).to(ltn.device))

# 3. AXIOMAS LÓGICOS
def define_geo_axioms(circles, squares, triangles):
    c = ltn.Variable("c", circles)
    s = ltn.Variable("s", squares)
    t = ltn.Variable("t", triangles)

    axioms = [
        # ANCORAGEM: ensina quem é quem
        Forall(c, is_circle(c)),
        Forall(c, Not(is_square(c))),
        Forall(c, Not(is_triangle(c))),
        Forall(s, is_square(s)),
        Forall(s, Not(is_circle(s))),
        Forall(s, Not(is_triangle(s))),
        Forall(t, is_triangle(t)),
        Forall(t, Not(is_circle(t))),
        Forall(t, Not(is_square(t))),

        # ANCORAGEM: direção das relações (a ordem inversa é falsa)
        Forall([s, c], Not(left_of(s, c))),
        Forall([c, t], Not(above(c, t))),

        # AXIOMAS DA TAREFA
        Forall([c, s], Implies(And(is_circle(c), is_square(s)), left_of(c, s))),
        Forall([t, c], Implies(And(is_triangle(t), is_circle(c)), above(t, c))),
        Forall([c, s], Implies(overlaps(c, s), overlaps(s, c))),
        Forall(c, Not(overlaps(c, c))),
        Forall(s, Not(overlaps(s, s))),
        Forall(t, Not(overlaps(t, t))),

        # AXIOMA EXTRA: quadrados e triângulos não se sobrepõem
        Forall([s, t], Implies(And(is_square(s), is_triangle(t)), Not(overlaps(s, t))))
    ]
    return sat_agg(*axioms)

# 4. TREINAMENTO (dados gerados uma única vez, e devolvidos)
def train_geometrical_model(epochs=300):
    parameters = []
    for pred in [is_circle, is_square, is_triangle, left_of, above, overlaps]:
        parameters += list(pred.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)

    circles, squares, triangles = generate_random_shapes(num_per_shape=10)

    print("\n=== TREINANDO AXIOMAS GEOMÉTRICOS ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat = define_geo_axioms(circles, squares, triangles)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Satisfação = {sat.item():.4f}")
    print(f"Epoch {epochs}: Satisfação final = {define_geo_axioms(circles, squares, triangles).item():.4f}")
    return circles, squares, triangles

# 5. TESTE EM NOVOS DADOS ALEATÓRIOS
def test_geometrical_model():
    print("\n=== TESTANDO EM NOVOS DADOS ALEATÓRIOS ===")
    test_circles, test_squares, test_triangles = generate_random_shapes(num_per_shape=5)
    c1 = ltn.Constant(test_circles[0])
    s1 = ltn.Constant(test_squares[0])
    t1 = ltn.Constant(test_triangles[0])

    print("--- Classificação (esperado: alto para a classe certa, baixo para as outras) ---")
    print(f"Q1: círculo é círculo?      {is_circle(c1).value.item():.4f}")
    print(f"Q2: quadrado é quadrado?    {is_square(s1).value.item():.4f}")
    print(f"Q3: triângulo é triângulo?  {is_triangle(t1).value.item():.4f}")
    print(f"Q4: quadrado é círculo?     {is_circle(s1).value.item():.4f}")

    print("--- Relações espaciais em pontos NUNCA vistos ---")
    print(f"Q5: círculo à esquerda do quadrado? {left_of(c1, s1).value.item():.4f}  (esperado alto)")
    print(f"Q6: quadrado à esquerda do círculo? {left_of(s1, c1).value.item():.4f}  (esperado baixo)")
    print(f"Q7: triângulo acima do círculo?     {above(t1, c1).value.item():.4f}  (esperado alto)")
    print(f"Q8: círculo acima do triângulo?     {above(c1, t1).value.item():.4f}  (esperado baixo)")

    print("--- Axiomas quantificados no conjunto de teste ---")
    c_t = ltn.Variable("c_t", test_circles)
    s_t = ltn.Variable("s_t", test_squares)
    t_t = ltn.Variable("t_t", test_triangles)
    q9 = Forall([c_t, s_t], left_of(c_t, s_t))
    q10 = Forall([t_t, c_t], above(t_t, c_t))
    q11 = Forall([s_t, t_t], Implies(And(is_square(s_t), is_triangle(t_t)), Not(overlaps(s_t, t_t))))
    print(f"Q9:  todos círculos à esquerda de todos quadrados? {q9.value.item():.4f}")
    print(f"Q10: todos triângulos acima de todos círculos?     {q10.value.item():.4f}")
    print(f"Q11: quadrados e triângulos não se sobrepõem?      {q11.value.item():.4f}")

train_geometrical_model(epochs=300)
test_geometrical_model()



EXTENSÃO: RELAÇÕES GEOMÉTRICAS EM UM PLANO 2D

=== TREINANDO AXIOMAS GEOMÉTRICOS ===
Epoch 0: Satisfação = 0.5128
Epoch 50: Satisfação = 0.8310
Epoch 100: Satisfação = 0.9119
Epoch 150: Satisfação = 0.9580
Epoch 200: Satisfação = 0.9798
Epoch 250: Satisfação = 0.9890
Epoch 300: Satisfação final = 0.9932

=== TESTANDO EM NOVOS DADOS ALEATÓRIOS ===
--- Classificação (esperado: alto para a classe certa, baixo para as outras) ---
Q1: círculo é círculo?      0.9996
Q2: quadrado é quadrado?    1.0000
Q3: triângulo é triângulo?  0.9999
Q4: quadrado é círculo?     0.0000
--- Relações espaciais em pontos NUNCA vistos ---
Q5: círculo à esquerda do quadrado? 1.0000  (esperado alto)
Q6: quadrado à esquerda do círculo? 0.0000  (esperado baixo)
Q7: triângulo acima do círculo?     1.0000  (esperado alto)
Q8: círculo acima do triângulo?     0.0000  (esperado baixo)
--- Axiomas quantificados no conjunto de teste ---
Q9:  todos círculos à esquerda de todos quadrados? 0.9999
Q10: todos triângulos acima 

In [29]:
torch.manual_seed(42)  # reprodutibilidade: mesmos resultados a cada execução
import math

print("\n" + "=" * 60)
print("EXTENSÃO: SÓLIDOS 3D E FÓRMULAS DE VOLUME")
print("=" * 60)

# Fórmulas de volume usadas:
#   Cone:           V = pi * r^2 * h / 3
#   Cilindro:       V = pi * r^2 * h
#   Paralelepípedo: V = a * b * c
#
# AXIOMAS EM FOL:
#  1. Ancoragem: Cone, Cilindro e Paralelepípedo são classes exclusivas
#  2. ∀x ∀y (Cilindro(x) ∧ Cone(y) ∧ mesma base e altura → MaiorVolume(x,y))
#  3. ∀x ∀y (MaiorVolume(x,y) → ¬MaiorVolume(y,x))                 (antissimetria)
#  4. ∀x ¬MaiorVolume(x,x)                                          (irreflexividade)
#  5. ∀x ∀y ∀z (MaiorVolume(x,y) ∧ MaiorVolume(y,z) → MaiorVolume(x,z))  (transitividade)
#  6. ∀x (Cone(x) → ∃y (Paralelepipedo(y) ∧ MaiorVolume(y,x)))
#  7. Pares decididos pela fórmula de volume: o de maior V é MaiorVolume do outro

# 1. DADOS: cada sólido = (largura, profundidade, altura, log-volume, preenchimento)
def make_features(w, d, h, vol):
    fill = vol / (w * d * h)
    return torch.stack([w, d, h, torch.log(vol), fill], dim=1)

def generate_random_solids(n=10):
    r = torch.rand(n, device=ltn.device) * 2.5 + 0.5      # raio em [0.5, 3.0]
    h = torch.rand(n, device=ltn.device) * 5.0 + 1.0      # altura em [1.0, 6.0]
    # cone e cilindro com o MESMO r e h (linha i de um corresponde à linha i do outro)
    cones = make_features(2 * r, 2 * r, h, math.pi * r ** 2 * h / 3)
    cylinders = make_features(2 * r, 2 * r, h, math.pi * r ** 2 * h)
    a = torch.rand(n, device=ltn.device) * 4.0 + 1.0
    b = torch.rand(n, device=ltn.device) * 4.0 + 1.0
    c = torch.rand(n, device=ltn.device) * 4.0 + 1.0
    boxes = make_features(a, b, c, a * b * c)
    return cones, cylinders, boxes

def make_volume_pairs(all_solids, n_pairs=60):
    # pares (maior, menor) decididos pela fórmula de volume
    i = torch.randint(0, all_solids.shape[0], (n_pairs,), device=ltn.device)
    j = torch.randint(0, all_solids.shape[0], (n_pairs,), device=ltn.device)
    keep = i != j
    a = all_solids[i[keep]]
    b = all_solids[j[keep]]
    a_bigger = (a[:, 3] > b[:, 3]).unsqueeze(1)
    big = torch.where(a_bigger, a, b)
    small = torch.where(a_bigger, b, a)
    return big, small

# 2. PREDICADOS
class SolidClassifier(nn.Module):
    # O tipo do sólido é decidido pela razão de preenchimento (volume / caixa envolvente),
    # que vem direto das fórmulas: cone = pi/12 ~ 0.26, cilindro = pi/4 ~ 0.79, paralelepípedo = 1.
    # A razão é centralizada e escalada para a rede aprender a faixa do cilindro (valor do meio).
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        x_val = x.value if hasattr(x, 'value') else x
        fill = (x_val[..., 4:5] - 0.68) * 10.0
        return self.net(fill).squeeze(-1)

class VolumeComparison(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x, y):
        x_val = x.value if hasattr(x, 'value') else x
        y_val = y.value if hasattr(y, 'value') else y
        diff = (x_val[..., 3] - y_val[..., 3]).unsqueeze(-1)   # log V(x) - log V(y)
        return self.net(diff).squeeze(-1)

is_cone = ltn.Predicate(SolidClassifier().to(ltn.device))
is_cylinder = ltn.Predicate(SolidClassifier().to(ltn.device))
is_box = ltn.Predicate(SolidClassifier().to(ltn.device))
bigger_vol = ltn.Predicate(VolumeComparison().to(ltn.device))

# 3. AXIOMAS
def define_solid_axioms(cones, cylinders, boxes, big, small):
    all_solids = torch.cat([cones, cylinders, boxes], dim=0)
    cone = ltn.Variable("cone", cones)
    cyl = ltn.Variable("cyl", cylinders)
    box = ltn.Variable("box", boxes)
    x = ltn.Variable("x", all_solids)
    y = ltn.Variable("y", all_solids)
    z = ltn.Variable("z", all_solids)
    big_v = ltn.Variable("big_v", big)
    small_v = ltn.Variable("small_v", small)

    axioms = [
        # ANCORAGEM: quem é quem
        Forall(cone, is_cone(cone)),
        Forall(cone, Not(is_cylinder(cone))),
        Forall(cone, Not(is_box(cone))),
        Forall(cyl, is_cylinder(cyl)),
        Forall(cyl, Not(is_cone(cyl))),
        Forall(cyl, Not(is_box(cyl))),
        Forall(box, is_box(box)),
        Forall(box, Not(is_cone(box))),
        Forall(box, Not(is_cylinder(box))),

        # FÓRMULA: V(cilindro) = 3 * V(cone) de mesma base e altura
        Forall(ltn.diag(cyl, cone),
               Implies(And(is_cylinder(cyl), is_cone(cone)), bigger_vol(cyl, cone))),

        # ANCORAGEM DA COMPARAÇÃO: pares decididos pela fórmula de volume
        Forall(ltn.diag(big_v, small_v), bigger_vol(big_v, small_v)),
        Forall(ltn.diag(big_v, small_v), Not(bigger_vol(small_v, big_v))),

        # PROPRIEDADES DA RELAÇÃO "MAIOR VOLUME"
        Forall(x, Not(bigger_vol(x, x))),
        Forall([x, y], Implies(bigger_vol(x, y), Not(bigger_vol(y, x)))),
        Forall([x, y, z], Implies(And(bigger_vol(x, y), bigger_vol(y, z)), bigger_vol(x, z))),

        # EXISTENCIAL: para todo cone existe um paralelepípedo maior
        Forall(cone, Exists(box, bigger_vol(box, cone)))
    ]
    return sat_agg(*axioms)

# 4. TREINAMENTO (dados gerados uma única vez)
def train_solid_model(epochs=800):
    parameters = []
    for pred in [is_cone, is_cylinder, is_box, bigger_vol]:
        parameters += list(pred.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.02)

    cones, cylinders, boxes = generate_random_solids(n=10)
    big, small = make_volume_pairs(torch.cat([cones, cylinders, boxes], dim=0))

    print("\n=== TREINANDO AXIOMAS DOS SÓLIDOS ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat = define_solid_axioms(cones, cylinders, boxes, big, small)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Satisfação = {sat.item():.4f}")
    final = define_solid_axioms(cones, cylinders, boxes, big, small)
    print(f"Epoch {epochs}: Satisfação final = {final.item():.4f}")

# 5. TESTE EM NOVOS SÓLIDOS
def test_solid_model():
    print("\n=== TESTANDO EM NOVOS SÓLIDOS ===")
    t_cones, t_cyls, t_boxes = generate_random_solids(n=5)
    c1 = ltn.Constant(t_cones[0])
    y1 = ltn.Constant(t_cyls[0])      # mesmo r e h do cone c1
    b1 = ltn.Constant(t_boxes[0])

    print("--- Classificação (esperado: alto na classe certa) ---")
    print(f"Q1: cone é cone?              {is_cone(c1).value.item():.4f}")
    print(f"Q2: cilindro é cilindro?      {is_cylinder(y1).value.item():.4f}")
    print(f"Q3: paralelepípedo é paralelepípedo? {is_box(b1).value.item():.4f}")
    print(f"Q4: cone é cilindro?          {is_cylinder(c1).value.item():.4f}  (esperado baixo)")

    print("--- Comparação de volumes em sólidos nunca vistos ---")
    print(f"Q5: cilindro > cone (mesma base/altura)? {bigger_vol(y1, c1).value.item():.4f}  (esperado alto)")
    print(f"Q6: cone > cilindro (mesma base/altura)? {bigger_vol(c1, y1).value.item():.4f}  (esperado baixo)")
    print(f"Q7: cone > ele mesmo?                    {bigger_vol(c1, c1).value.item():.4f}  (esperado baixo)")
    v_c = math.exp(t_cones[0, 3].item())
    v_b = math.exp(t_boxes[0, 3].item())
    print(f"Q8: V(cone)={v_c:.2f}, V(paralelepípedo)={v_b:.2f}")
    print(f"    paralelepípedo > cone? {bigger_vol(b1, c1).value.item():.4f}")
    print(f"    cone > paralelepípedo? {bigger_vol(c1, b1).value.item():.4f}")

    print("--- Axiomas quantificados no conjunto de teste ---")
    all_t = torch.cat([t_cones, t_cyls, t_boxes], dim=0)
    tc = ltn.Variable("tc", t_cones)
    ty = ltn.Variable("ty", t_cyls)
    x = ltn.Variable("tx", all_t)
    y = ltn.Variable("ty2", all_t)
    z = ltn.Variable("tz", all_t)
    q9 = Forall(ltn.diag(ty, tc), bigger_vol(ty, tc))
    q10 = Forall([x, y, z], Implies(And(bigger_vol(x, y), bigger_vol(y, z)), bigger_vol(x, z)))
    q11 = Forall([x, y], Implies(bigger_vol(x, y), Not(bigger_vol(y, x))))
    print(f"Q9:  todo cilindro > cone de mesma base/altura? {q9.value.item():.4f}")
    print(f"Q10: transitividade de MaiorVolume?             {q10.value.item():.4f}")
    print(f"Q11: antissimetria de MaiorVolume?              {q11.value.item():.4f}")

train_solid_model(epochs=800)
test_solid_model()



EXTENSÃO: SÓLIDOS 3D E FÓRMULAS DE VOLUME

=== TREINANDO AXIOMAS DOS SÓLIDOS ===
Epoch 0: Satisfação = 0.5712
Epoch 50: Satisfação = 0.8907
Epoch 100: Satisfação = 0.9054
Epoch 150: Satisfação = 0.9107
Epoch 200: Satisfação = 0.9138
Epoch 250: Satisfação = 0.9170
Epoch 300: Satisfação = 0.9207
Epoch 350: Satisfação = 0.9245
Epoch 400: Satisfação = 0.9277
Epoch 450: Satisfação = 0.9299
Epoch 500: Satisfação = 0.9312
Epoch 550: Satisfação = 0.9320
Epoch 600: Satisfação = 0.9326
Epoch 650: Satisfação = 0.9329
Epoch 700: Satisfação = 0.9332
Epoch 750: Satisfação = 0.9333
Epoch 800: Satisfação final = 0.9335

=== TESTANDO EM NOVOS SÓLIDOS ===
--- Classificação (esperado: alto na classe certa) ---
Q1: cone é cone?              0.9996
Q2: cilindro é cilindro?      0.9966
Q3: paralelepípedo é paralelepípedo? 0.9985
Q4: cone é cilindro?          0.0006  (esperado baixo)
--- Comparação de volumes em sólidos nunca vistos ---
Q5: cilindro > cone (mesma base/altura)? 1.0000  (esperado alto)
Q6: co

In [30]:
torch.manual_seed(42)  # reprodutibilidade: mesmos resultados a cada execução
print("\n" + "=" * 60)
print("EXTENSÃO 2D AVANÇADA: AXIOMAS MAIS ELABORADOS")
print("=" * 60)

# Reaproveita ShapeClassifier, LearnableSpatialRelation, generate_random_shapes
# e os conectivos/quantificadores definidos nas células anteriores.
#
# NOVOS PREDICADOS: RightOf(x,y) (inverso de LeftOf) e Below(x,y) (inverso de Above)
#
# AXIOMAS EM FOL (além dos axiomas básicos e do extra já usados antes):
#  1. Exclusão mútua das classes:      ∀x ¬(Circle(x) ∧ Square(x)), idem Circle/Triangle e Square/Triangle
#  2. Transitividade de LeftOf:        ∀x∀y∀z (LeftOf(x,y) ∧ LeftOf(y,z) → LeftOf(x,z))
#  3. Transitividade de Above:         ∀x∀y∀z (Above(x,y) ∧ Above(y,z) → Above(x,z))
#  4. Antissimetria de LeftOf/Above:   ∀x∀y (LeftOf(x,y) → ¬LeftOf(y,x)), idem Above
#  5. Irreflexividade:                 ∀x ¬LeftOf(x,x), ∀x ¬Above(x,x)
#  6. Relações inversas:               ∀x∀y (LeftOf(x,y) ↔ RightOf(y,x)), ∀x∀y (Above(x,y) ↔ Below(y,x))
#  7. Existencial (sugerido no PDF):   ∀c (Circle(c) → ∃s (Square(s) ∧ LeftOf(c,s)))
#  8. Ancoragem pela "fórmula" das coordenadas: se x_a < x_b então LeftOf(a,b);
#     se y_a > y_b então Above(a,b) (pares de figuras quaisquer)

def right_of_features(x, y):
    return (x[..., 0] - y[..., 0]).unsqueeze(-1)

def below_features(x, y):
    return (y[..., 1] - x[..., 1]).unsqueeze(-1)

is_circle2 = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_square2 = ltn.Predicate(ShapeClassifier().to(ltn.device))
is_triangle2 = ltn.Predicate(ShapeClassifier().to(ltn.device))
left_of2 = ltn.Predicate(LearnableSpatialRelation(left_of_features).to(ltn.device))
right_of2 = ltn.Predicate(LearnableSpatialRelation(right_of_features).to(ltn.device))
above2 = ltn.Predicate(LearnableSpatialRelation(above_features).to(ltn.device))
below2 = ltn.Predicate(LearnableSpatialRelation(below_features).to(ltn.device))
overlaps2 = ltn.Predicate(LearnableSpatialRelation(distance_features).to(ltn.device))

def make_coordinate_pairs(all_figs, axis, n_pairs=80):
    # devolve (menor, maior) segundo a coordenada escolhida (0 = x, 1 = y)
    i = torch.randint(0, all_figs.shape[0], (n_pairs,), device=ltn.device)
    j = torch.randint(0, all_figs.shape[0], (n_pairs,), device=ltn.device)
    keep = i != j
    a = all_figs[i[keep]]
    b = all_figs[j[keep]]
    a_lower = (a[:, axis] < b[:, axis]).unsqueeze(1)
    lo = torch.where(a_lower, a, b)
    hi = torch.where(a_lower, b, a)
    return lo, hi

def define_geo_axioms2(circles, squares, triangles, pairs_x, pairs_y):
    all_figs = torch.cat([circles, squares, triangles], dim=0)
    c = ltn.Variable("c", circles)
    s = ltn.Variable("s", squares)
    t = ltn.Variable("t", triangles)
    x = ltn.Variable("x", all_figs)
    y = ltn.Variable("y", all_figs)
    z = ltn.Variable("z", all_figs)
    left_v = ltn.Variable("left_v", pairs_x[0])     # menor x  -> está à esquerda
    right_v = ltn.Variable("right_v", pairs_x[1])
    low_v = ltn.Variable("low_v", pairs_y[0])       # menor y  -> está abaixo
    high_v = ltn.Variable("high_v", pairs_y[1])

    axioms = [
        # ANCORAGEM: classes
        Forall(c, is_circle2(c)), Forall(c, Not(is_square2(c))), Forall(c, Not(is_triangle2(c))),
        Forall(s, is_square2(s)), Forall(s, Not(is_circle2(s))), Forall(s, Not(is_triangle2(s))),
        Forall(t, is_triangle2(t)), Forall(t, Not(is_circle2(t))), Forall(t, Not(is_square2(t))),

        # ANCORAGEM: relações pelas coordenadas
        Forall(ltn.diag(left_v, right_v), left_of2(left_v, right_v)),
        Forall(ltn.diag(left_v, right_v), Not(left_of2(right_v, left_v))),
        Forall(ltn.diag(high_v, low_v), above2(high_v, low_v)),
        Forall(ltn.diag(high_v, low_v), Not(above2(low_v, high_v))),

        # AXIOMAS BÁSICOS DA TAREFA
        Forall([c, s], Implies(And(is_circle2(c), is_square2(s)), left_of2(c, s))),
        Forall([t, c], Implies(And(is_triangle2(t), is_circle2(c)), above2(t, c))),
        Forall([c, s], Implies(overlaps2(c, s), overlaps2(s, c))),
        Forall(x, Not(overlaps2(x, x))),

        # AXIOMA EXTRA ANTERIOR
        Forall([s, t], Implies(And(is_square2(s), is_triangle2(t)), Not(overlaps2(s, t)))),

        # NOVOS: exclusão mútua das classes
        Forall(x, Not(And(is_circle2(x), is_square2(x)))),
        Forall(x, Not(And(is_circle2(x), is_triangle2(x)))),
        Forall(x, Not(And(is_square2(x), is_triangle2(x)))),

        # NOVOS: transitividade
        Forall([x, y, z], Implies(And(left_of2(x, y), left_of2(y, z)), left_of2(x, z))),
        Forall([x, y, z], Implies(And(above2(x, y), above2(y, z)), above2(x, z))),

        # NOVOS: antissimetria e irreflexividade
        Forall([x, y], Implies(left_of2(x, y), Not(left_of2(y, x)))),
        Forall([x, y], Implies(above2(x, y), Not(above2(y, x)))),
        Forall(x, Not(left_of2(x, x))),
        Forall(x, Not(above2(x, x))),

        # NOVOS: relações inversas (equivalências)
        Forall([x, y], Implies(left_of2(x, y), right_of2(y, x))),
        Forall([x, y], Implies(right_of2(y, x), left_of2(x, y))),
        Forall([x, y], Implies(above2(x, y), below2(y, x))),
        Forall([x, y], Implies(below2(y, x), above2(x, y))),

        # NOVO: existencial sugerido no PDF
        Forall(c, Implies(is_circle2(c), Exists(s, And(is_square2(s), left_of2(c, s)))))
    ]
    return sat_agg(*axioms)

def train_geometrical_model2(epochs=500):
    parameters = []
    for pred in [is_circle2, is_square2, is_triangle2, left_of2, right_of2, above2, below2, overlaps2]:
        parameters += list(pred.model.parameters())
    optimizer = torch.optim.Adam(parameters, lr=0.01)

    circles, squares, triangles = generate_random_shapes(num_per_shape=20)
    all_figs = torch.cat([circles, squares, triangles], dim=0)
    pairs_x = make_coordinate_pairs(all_figs, axis=0)
    pairs_y = make_coordinate_pairs(all_figs, axis=1)

    print("\n=== TREINANDO AXIOMAS 2D AVANÇADOS ===")
    for epoch in range(epochs):
        optimizer.zero_grad()
        sat = define_geo_axioms2(circles, squares, triangles, pairs_x, pairs_y)
        loss = 1 - sat
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Satisfação = {sat.item():.4f}")
    final = define_geo_axioms2(circles, squares, triangles, pairs_x, pairs_y)
    print(f"Epoch {epochs}: Satisfação final = {final.item():.4f}")

def test_geometrical_model2():
    print("\n=== TESTANDO EM NOVOS DADOS ALEATÓRIOS (2D avançado) ===")
    t_circles, t_squares, t_triangles = generate_random_shapes(num_per_shape=5)
    c1 = ltn.Constant(t_circles[0])
    s1 = ltn.Constant(t_squares[0])
    t1 = ltn.Constant(t_triangles[0])

    print(f"Pontos de teste: círculo=({t_circles[0,0].item():.2f}, {t_circles[0,1].item():.2f}), "
          f"quadrado=({t_squares[0,0].item():.2f}, {t_squares[0,1].item():.2f}), "
          f"triângulo=({t_triangles[0,0].item():.2f}, {t_triangles[0,1].item():.2f})")
    print("--- Classificação ---")
    print(f"Q1: círculo é círculo?     {is_circle2(c1).value.item():.4f}  (esperado alto)")
    print(f"Q2: quadrado é quadrado?   {is_square2(s1).value.item():.4f}  (esperado alto)")
    print(f"Q3: triângulo é triângulo? {is_triangle2(t1).value.item():.4f}  (esperado alto)")
    print(f"Q4: círculo é quadrado?    {is_square2(c1).value.item():.4f}  (esperado baixo)")

    print("--- Relações inversas em pontos nunca vistos ---")
    print(f"Q5: LeftOf(círculo, quadrado)  = {left_of2(c1, s1).value.item():.4f}  (esperado alto)")
    print(f"Q6: RightOf(quadrado, círculo) = {right_of2(s1, c1).value.item():.4f}  (esperado alto)")
    print(f"Q7: Above(triângulo, círculo)  = {above2(t1, c1).value.item():.4f}  (esperado alto)")
    print(f"Q8: Below(círculo, triângulo)  = {below2(c1, t1).value.item():.4f}  (esperado alto)")

    print("--- Par triângulo/quadrado (a resposta correta vem das coordenadas) ---")
    tx, sx = t_triangles[0, 0].item(), t_squares[0, 0].item()
    ty, sy = t_triangles[0, 1].item(), t_squares[0, 1].item()
    print(f"Q9:  triângulo x={tx:.2f}, quadrado x={sx:.2f} -> triângulo à esquerda? "
          f"{left_of2(t1, s1).value.item():.4f}  (verdade: {tx < sx})")
    print(f"Q10: triângulo y={ty:.2f}, quadrado y={sy:.2f} -> triângulo acima? "
          f"{above2(t1, s1).value.item():.4f}  (verdade: {ty > sy})")

    print("--- Axiomas quantificados no conjunto de teste ---")
    all_t = torch.cat([t_circles, t_squares, t_triangles], dim=0)
    tc = ltn.Variable("tc", t_circles)
    ts = ltn.Variable("ts", t_squares)
    x = ltn.Variable("tx", all_t)
    y = ltn.Variable("ty", all_t)
    z = ltn.Variable("tz", all_t)
    q11 = Forall([x, y, z], Implies(And(left_of2(x, y), left_of2(y, z)), left_of2(x, z)))
    q12 = Forall([x, y, z], Implies(And(above2(x, y), above2(y, z)), above2(x, z)))
    q13 = Forall([x, y], Implies(left_of2(x, y), Not(left_of2(y, x))))
    q14 = Forall([x, y], Implies(left_of2(x, y), right_of2(y, x)))
    q15 = Forall(tc, Exists(ts, left_of2(tc, ts)))
    print(f"Q11: transitividade de LeftOf?           {q11.value.item():.4f}")
    print(f"Q12: transitividade de Above?            {q12.value.item():.4f}")
    print(f"Q13: antissimetria de LeftOf?            {q13.value.item():.4f}")
    print(f"Q14: LeftOf(x,y) -> RightOf(y,x)?        {q14.value.item():.4f}")
    print(f"Q15: todo círculo tem um quadrado à direita? {q15.value.item():.4f}")

train_geometrical_model2(epochs=500)
test_geometrical_model2()



EXTENSÃO 2D AVANÇADA: AXIOMAS MAIS ELABORADOS

=== TREINANDO AXIOMAS 2D AVANÇADOS ===
Epoch 0: Satisfação = 0.5617
Epoch 50: Satisfação = 0.8342
Epoch 100: Satisfação = 0.9137
Epoch 150: Satisfação = 0.9484
Epoch 200: Satisfação = 0.9592
Epoch 250: Satisfação = 0.9638
Epoch 300: Satisfação = 0.9672
Epoch 350: Satisfação = 0.9710
Epoch 400: Satisfação = 0.9750
Epoch 450: Satisfação = 0.9784
Epoch 500: Satisfação final = 0.9818

=== TESTANDO EM NOVOS DADOS ALEATÓRIOS (2D avançado) ===
Pontos de teste: círculo=(2.87, 3.53), quadrado=(7.98, 1.85), triângulo=(9.50, 9.06)
--- Classificação ---
Q1: círculo é círculo?     0.9983  (esperado alto)
Q2: quadrado é quadrado?   0.9998  (esperado alto)
Q3: triângulo é triângulo? 1.0000  (esperado alto)
Q4: círculo é quadrado?    0.0000  (esperado baixo)
--- Relações inversas em pontos nunca vistos ---
Q5: LeftOf(círculo, quadrado)  = 1.0000  (esperado alto)
Q6: RightOf(quadrado, círculo) = 1.0000  (esperado alto)
Q7: Above(triângulo, círculo)  = 1.0